# Stencil Computation for 2D Gray-Scott Using Julia

In [ ]:
# Pick your hardware: :CUDA (NVIDIA), :AMDGPU (AMD), :Metal (Apple), or :CPU
DEV = :CUDA


# 1. Setup

The setup is the same as in the GPU introduction, which explains it. The cells below activate the project environment, install and load the packages, and select the backend; set `DEV` in the **second cell** (run it before `Pkg.add`) to match your hardware.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)

In [ ]:
packages = ["BenchmarkTools", "ProgressMeter", "Plots", "ColorTypes", "KernelAbstractions"]
if DEV != :CPU
    push!(packages, string(DEV))
end
Pkg.add(packages)


In [ ]:
using BenchmarkTools, ProgressMeter, Plots, ColorTypes, KernelAbstractions


In [ ]:
Pkg.status()

## 1.1. Selecting a backend

Each vendor has its own Julia package (CUDA.jl, AMDGPU.jl, Metal.jl). `Pkg.add` uses `string(DEV)` when `DEV != :CPU`. The cell below runs `@eval using $(DEV)` and sets `ArrayKA` and `Backend`.


In [ ]:
if DEV != :CPU
    @eval using $(DEV)
end

if DEV == :CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
elseif DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
else
    error("Unknown DEV = $DEV. Use :CUDA, :AMDGPU, :Metal, or :CPU.")
end


# 2. Gray-Scott Reaction-Diffusion Model

The Gray-Scott Reaction-Diffusion model is a mathematical model used to describe how two interacting substances, typically denoted as $U$ and $V$, evolve over time in a spatial domain. This model captures the combined effects of chemical reactions and diffusion, leading to complex pattern formations such as spots, stripes, and waves, which resemble natural phenomena like animal skin patterns or chemical reactions in a petri dish.

## 2.1. Equations

The evolution of $ U $ and $ V $ over time is described by the following partial differential equations (PDEs):

$$
\frac{\partial U}{\partial t} = D_u \nabla^2 U - UV^2 + F(1 - U)
$$

$$
\frac{\partial V}{\partial t} = D_v \nabla^2 V + UV^2 - (F + k)V
$$

- **$D_u$**: Diffusion coefficient of $U$, controlling its spread rate.
- **$D_v$**: Diffusion coefficient of $V$, controlling its spread rate.
- **$F$**: Feed rate of  $U$, representing the replenishment of  $U$ in the system.
- **$ k $**: Kill rate of $V$, representing the removal of $V$ from the system.
- **$ \nabla^2 $**: Laplacian operator, representing the diffusion term.

## 2.2. 2D 5-Point Stencil

The 2D 5-point stencil is used to compute the diffusion terms ($ D_u \nabla^2 U $ and $ D_v \nabla^2 V $) in the Gray-Scott reaction-diffusion equations. It helps model how the substances $ U $ and $V $ spread through the 2D spatial domain by capturing the local flow of concentration between neighboring points.

<img src="img/stencil_image1.png" alt="Input grid, 5-point stencil operator, and output grid" style="max-width:680px; width:100%; height:auto; display:block;" />

The left grid is the input field. The center shows the five cells the stencil touches at $(i, j)$: the point itself and its four orthogonal neighbors. The right grid is where the updated value is written. Every interior point of $U$ and $V$ is updated this way each time step.

<img src="img/stencil_image2.webp" alt="5-point stencil $S_{i,j}$ centered on $v_{i,j}$ with four neighbors" style="max-width:300px; width:100%; height:auto; display:block;" />

On the mesh, $S_{i,j}$ is that same cross: the black center $v_{i,j}$ and the four yellow neighbors. Points on the domain edge are missing a neighbor, so the code below skips the boundary (ghost) cells and applies the stencil only for $2 \le x \le n-1$ and $2 \le y \le n-1$.

### 2.3. Mathematical Formulation

The Laplacian of a function $ u $ at point $(i, j)$ using the 2D 5-point stencil is approximated as:

$$
\nabla^2 u_{i, j} \approx \frac{1}{h^2} \left( u_{i-1, j} + u_{i+1, j} - 2 \cdot u_{i, j} \right) + \frac{1}{h^2} \left( u_{i, j-1} + u_{i, j+1} - 2 \cdot u_{i, j} \right)
$$

- **$ h $** is the grid spacing (`dx` in the code).
- Each term is the centered second difference along one axis; adding them gives $\nabla^2 u$ from the five stencil points above.

# 3. From CPU to KernelAbstractions

Section 3.1 gives a reference CPU implementation. Exercise 6 describes how to port it to KernelAbstractions, and you complete the port in §3.2. A complete solution is in `proposed-solution/exercise_6_gray_scott.ipynb`. An optional follow-up in `proposed-solution/exercise_6_gray_scott_tuned.ipynb` assigns each work item a **`P` × `Q`** block of cells (`P` rows, `Q` columns along the grid, not the PDE parameters `F` or `k`); see that notebook for what `P` and `Q` mean there.

## 3.1. Reference CPU implementation
We use Float32 for the fields and PDE parameters on CPU and GPU. Single precision is much faster on GPUs; Apple GPUs do not support Float64 in kernels.


This version is the baseline for correctness and performance. Tutorial code uses `my_gs_cpu!`, `my_simulate_gs_cpu`, and (after Exercise 6) `my_gs_gpu_kernel!` / `my_simulate_gs_gpu`, with `my_laplacian` shared by CPU and GPU. You will port `my_gs_cpu!` and `my_simulate_gs_cpu` to KernelAbstractions in §3.2.

In [ ]:
# Reference CPU implementation — start from this, then port it to KernelAbstractions below.

function my_laplacian(A, x, y, dx)
    @inbounds DAx = (A[x-1, y] - (2 * A[x, y]) + A[x+1, y]) / dx
    @inbounds DAy = (A[x, y-1] - (2 * A[x, y]) + A[x, y+1]) / dx
    return DAx + DAy
end

function my_gs_cpu!(U, V, D_u, D_v, F, k, dt, dx, nx, ny, noise)
    ΔU = zeros(eltype(U), nx, ny)
    ΔV = zeros(eltype(U), nx, ny)

    for x in 2:(nx-1), y in 2:(ny-1)
        @inbounds ΔU[x, y] = my_laplacian(U, x, y, dx)
        @inbounds ΔV[x, y] = my_laplacian(V, x, y, dx)
    end

    UVV = U .* V .* V
    N = noise == 0 ? zeros(eltype(U), nx, ny) : noise .* randn(eltype(U), nx, ny)
    U .+= (D_u .* ΔU .- UVV .+ F .* (1 .- U) .+ N) .* dt
    V .+= (D_v .* ΔV .+ UVV .- (F .+ k) .* V) .* dt
    return
end

function my_simulate_gs_cpu(n = 500 #=Grid size=#, nsteps = 10000 #=Number of iterations=# ;
                         init_multi = 0.5,
                         noise = 0.0f0,
                         D_u = 0.16f0,
                         D_v = 0.08f0,
                         F = 0.035f0,
                         k = 0.06f0,
                         dt = 1.0f0,
                         dx = 1.0f0,
                         plot_every = 500)

    U = Float32.(rand(0.0:0.001:0.1, n, n))
    V = zeros(Float32, n, n)

    n_mid = div(n, 2)
    U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(init_multi)
    V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(0.5 * init_multi)

    prog = Progress(div(nsteps, plot_every))
    plt = Plots.plot(title="Gray-Scott Model (CPU)", xlabel="X", ylabel="Y")

    anim = Plots.@animate for T in 1:div(nsteps, plot_every)
        for t in 1:plot_every
            my_gs_cpu!(U, V, D_u, D_v, F, k, dt, dx, n, n, noise)
        end
        next!(prog)
        Plots.heatmap!(plt, U, color=cgrad([:black, :blue, :white]), clims=extrema(U), title="Gray-Scott Model (CPU)")
    end
    gif(anim, "gs_cpu_serial.gif"; fps=10)
end

### 3.2. Timing the update

`my_simulate_gs_cpu` spends much of its time building the animation. The next cell times only the update, `my_gs_cpu!`, for 20 steps on grids from \(64^2\) to \(512^2\).

In [ ]:
sizes = [64, 128, 256, 512]
gs_steps = 20
t_cpu = Float64[]
for n in sizes
    U, V = rand(Float32, n, n), rand(Float32, n, n)
    push!(t_cpu, @belapsed begin
        for _ in 1:gs_steps
            my_gs_cpu!($U, $V, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, $n, $n, 0.0f0)
        end
    end)
end

tlabel(v) = begin
    x, u = v < 1e-6 ? (v * 1e9, "ns") : v < 1e-3 ? (v * 1e6, "µs") : v < 1 ? (v * 1e3, "ms") : (v, "s")
    r = round(x; sigdigits = 2)
    string(isinteger(r) ? Int(r) : r, " ", u)
end

Plots.plot(sizes, t_cpu;
    label = ["CPU"],
    xlabel = "grid size n", ylabel = "time",
    xscale = :log2, yscale = :log10,
    xticks = (sizes, string.(sizes)),
    yformatter = tlabel, marker = :circle, legend = :topleft)

In [ ]:
@time my_simulate_gs_cpu(500, 10000)

## 1.1. Selecting a backend

Each vendor has its own Julia package (CUDA.jl, AMDGPU.jl, Metal.jl). `Pkg.add` uses `string(DEV)` when `DEV != :CPU`. The cell below runs `@eval using $(DEV)` and sets `ArrayKA` and `Backend`.


## 3.3. Code skeleton

Complete each `TODO` following the tasks in Exercise 6. Code that is unchanged from the CPU version is already filled in.

In [ ]:
# Port the reference CPU implementation (Exercise 6).
# Same-as-CPU logic is filled in; TODOs mark what you must change.
# Full answer: proposed-solution/exercise_6_gray_scott.ipynb

# TODO (1): use `@kernel function` here instead of `function`
function my_gs_gpu_kernel!(Uout, Vout, U, V, D_u, D_v, F, k, dt, dx, nx, ny, noise)
    # TODO: mark U and V as `@Const` in the `@kernel` signature
    # TODO (3): allocate register memory in advance with `@private` (e.g. ΔU = @private Float32 1)
    # for ΔU, ΔV, UVV, Nr, DU, DV — then use ΔU[1], DU[1], … below

    # TODO (2): use `@index(Global, NTuple)` for (x, y), then an `if` to skip ghost cells
    # (replace this CPU-style loop)
    for x in 2:(nx-1), y in 2:(ny-1)

        # Same PDE as CPU, but into register slots (after you add `@private` above)
        ΔU[1] = my_laplacian(U, x, y, dx)
        ΔV[1] = my_laplacian(V, x, y, dx)

        UVV[1] = @inbounds U[x, y] * (V[x, y] ^ 2)
        Nr[1] = noise == 0 ? zero(eltype(U)) : noise * randn(Float32)
        DU[1] = @inbounds D_u * ΔU[1] - UVV[1] + F * (1 - U[x, y]) + Nr[1]
        DV[1] = @inbounds D_v * ΔV[1] + UVV[1] - (F + k) * V[x, y]

        # TODO (4): write to Uout/Vout (do not update U/V in place)
        @inbounds Uout[x, y] = U[x, y] + DU[1] * dt
        @inbounds Vout[x, y] = V[x, y] + DV[1] * dt
    end
end

function my_simulate_gs_gpu(n = 500 #=Grid size=#, nsteps = 10000 #=Number of iterations=# ;
                        init_multi = 0.5,
                        noise = 0.0f0,
                        D_u = 0.16f0,
                        D_v = 0.08f0,
                        F = 0.035f0,
                        k = 0.06f0,
                        dt = 1.0f0,
                        dx = 1.0f0,
                        plot_every = 500)

    # Same host init as my_simulate_gs_cpu
    U = Float32.(rand(0.0:0.001:0.1, n, n))
    V = zeros(Float32, n, n)

    n_mid = div(n, 2)
    U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(init_multi)
    V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= Float32(0.5 * init_multi)

    # TODO (5): allocate U1, V1, U2, V2 on the device (Float32, n × n) and copyto! initial U, V into both pairs

    # TODO (6): use my_gs_gpu_kernel!(Backend, (cld(n, 16), cld(n, 16))) to build the launcher kernel
    # TODO (7): set readU, readV, writeU, writeV = $U1, $V1, $U2, $V2 (swap each step)

    # Same progress / plot setup as CPU
    prog = Progress(div(nsteps, plot_every))
    plt = Plots.plot(title="Gray-Scott Model", xlabel="X", ylabel="Y")

    anim = Plots.@animate for t in 1:div(nsteps, plot_every)
        for _ in 1:plot_every
            # TODO (8): kernel(writeU, writeV, readU, readV, …; ndrange=(n, n)) then swap read/write
        end

        # TODO (9): copyto! host U from readU; KernelAbstractions.synchronize(Backend)

        # Same plotting as CPU
        next!(prog)
        Plots.heatmap!(plt, U, color=cgrad([:black, :blue, :white]), clims=extrema(U), title="Gray-Scott Model")
    end
    gif(anim, "gs_ka.gif"; fps=10)
end

In [ ]:
@time my_simulate_gs_gpu(500, 10000)

# 4. Experiments

## 4.1. The same grids on the GPU

The CPU update was timed in §3.1. Once `my_gs_gpu_kernel!` from Exercise 6 runs, the next cell times it for the same 20 steps and draws it beside that curve. The `@time` measurements on `my_simulate_gs_gpu` include the animation, so they answer a different question.

After that, vary the noise and the parameters of the PDE. The cells below are a few such runs.

In [ ]:
t_gpu = Float64[]
for n in sizes
    Uhost = rand(Float32, n, n)
    Vhost = rand(Float32, n, n)
    U1 = KernelAbstractions.allocate(Backend, Float32, n, n)
    V1 = KernelAbstractions.allocate(Backend, Float32, n, n)
    U2 = KernelAbstractions.allocate(Backend, Float32, n, n)
    V2 = KernelAbstractions.allocate(Backend, Float32, n, n)
    KernelAbstractions.copyto!(Backend, U1, Uhost)
    KernelAbstractions.copyto!(Backend, V1, Vhost)
    KernelAbstractions.copyto!(Backend, U2, Uhost)
    KernelAbstractions.copyto!(Backend, V2, Vhost)
    k! = my_gs_gpu_kernel!(Backend, (cld(n, 16), cld(n, 16)))
    push!(t_gpu, @belapsed begin
        readU, readV, writeU, writeV = $U1, $V1, $U2, $V2
        for _ in 1:gs_steps
            $k!(writeU, writeV, readU, readV, 0.16f0, 0.08f0, 0.035f0, 0.06f0, 1.0f0, 1.0f0, $n, $n, 0.0f0; ndrange = ($n, $n))
            readU, readV, writeU, writeV = writeU, writeV, readU, readV
        end
        KernelAbstractions.synchronize($Backend)
    end)
end

Plots.plot(sizes, [t_cpu t_gpu];
    label = ["CPU" "GPU kernel"],
    xlabel = "grid size n", ylabel = "time",
    xscale = :log2, yscale = :log10,
    xticks = (sizes, string.(sizes)),
    yformatter = tlabel, marker = :circle, legend = :topleft)

In [ ]:
# Add a lot more noise
@time my_simulate_gs_gpu(500, 10000; noise=0.03f0)

In [ ]:
# Tweak some parameters
@time my_simulate_gs_gpu(500, 50000; k=.06054f0, F=.07528f0, noise=0.0001f0, init_multi=1.0f0, plot_every=2000)